# Prepare Dataset
Input:
- Directory containing HLW camera trap MP4 videos for each location (e.g., H03, H23)
- COCO annotations with frame-video mapping filenames

Output:
- Stratified Train/val/test split.json

In [2]:
import os
import sys
import json
import copy
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import supervision as sv

sys.path.insert(0, str(Path("..").resolve()))
from hlwdetector.visualization.video_annotator import VideoAnnotator

sys.path.insert(0, str(Path("../utilities").resolve()))
from video_dataset_prep_tools import compute_split_statistics, site_video_split, split_report, extract_frames_from_dir, remove_multi_bird_frames, extract_single_video
from annotation_converter import AnnotationConverter

## Prepare Images

### Extract Frames
All frames should be placed in a dedicated flat directory with the naming convention SITE_VIDEO_FRAME.png, e.g., H23_IMG_0225_frame_000009.png

In [2]:
sites = ["h42"]
coco_ann_files = {}
for site in sites:
    coco_ann_files[site] = Path(f"../data/annotations/{site}_coco.json")

In [3]:
out_dir = Path("../data/frames")
for site in sites:
    video_dir = Path(f"../data/videos/{site}")
    #extract_frames_from_dir(site_name=site.upper(), video_dir=video_dir, out_dir=out_dir)

### Sanity Check: Annotations vs. Extracted Frames

Load `master_coco.json` (all sites) and sample 3 annotated frames per site, grouped by the `SITE_` prefix of the frame filename. Plot the ground truth bounding boxes on each frame from the flat `data/frames` directory. This verifies that frame filenames in the COCO annotations match the extracted frames on disk.

In [22]:
N_SAMPLES = 3
random.seed(405)

master_coco_path = Path("../data/annotations/master_coco.json")
images_dir = Path("../data/frames")

with open(master_coco_path) as f:
    coco = json.load(f)

# Build stem -> sv.Detections from COCO [x,y,w,h] annotations
id_to_stem = {img["id"]: Path(img["file_name"]).stem for img in coco["images"]}
stem_to_boxes: dict[str, list] = {}
for ann in coco["annotations"]:
    stem = id_to_stem.get(ann["image_id"])
    if stem is None:
        continue
    x, y, w, h = ann["bbox"]
    stem_to_boxes.setdefault(stem, []).append([x, y, x + w, y + h])
gt_detections = {
    stem: sv.Detections(
        xyxy=np.array(boxes, dtype=np.float32),
        class_id=np.zeros(len(boxes), dtype=int),
    )
    for stem, boxes in stem_to_boxes.items()
}

# Group annotated frames by site (filename convention: SITE_VIDEO_frame_NNNNNN.png)
annotated_ids = {ann["image_id"] for ann in coco["annotations"]}
by_site: dict[str, list] = {}
for img in coco["images"]:
    if img["id"] in annotated_ids:
        by_site.setdefault(img["file_name"].split("_")[0], []).append(img)

# Sample N_SAMPLES annotated frames per site
samples = [
    (site, img)
    for site in sorted(by_site)
    for img in random.sample(by_site[site], min(N_SAMPLES, len(by_site[site])))
]

annotator = VideoAnnotator(
    images_dir=str(images_dir),
    gt_detections=gt_detections,
    class_names=["bird"],
    image_files=[images_dir / img["file_name"] for _, img in samples
                 if (images_dir / img["file_name"]).exists()],
)

n_rows = len(by_site)
fig, axes = plt.subplots(n_rows, N_SAMPLES, figsize=(N_SAMPLES * 6, n_rows * 5), squeeze=False)
all_ok = True

for i, (site, img_meta) in enumerate(samples):
    ax = axes[i // N_SAMPLES][i % N_SAMPLES]
    frame_path = images_dir / img_meta["file_name"]

    if not frame_path.exists():
        ax.text(0.5, 0.5, f"FILE NOT FOUND:\n{img_meta['file_name']}",
                ha="center", va="center", transform=ax.transAxes, color="red", fontsize=9)
        ax.set_title(f"[{site}] MISSING", color="red")
        ax.axis("off")
        all_ok = False
        continue

    annotated_bgr = annotator.annotate_single_frame(str(frame_path))
    n_birds = len(gt_detections.get(frame_path.stem, []))
    ax.imshow(annotated_bgr[:, :, ::-1])  # BGR -> RGB
    ax.set_title(f"[{site}] {img_meta['file_name']}\n{n_birds} bird(s)", fontsize=8)
    ax.axis("off")

# Hide unused axes (sites with fewer than N_SAMPLES annotated frames)
for j in range(len(samples), n_rows * N_SAMPLES):
    axes[j // N_SAMPLES][j % N_SAMPLES].axis("off")

plt.tight_layout()
plt.show()

if all_ok:
    print("✓ All sampled frames found on disk")
else:
    print("✗ Some frames were not found. Check that extraction completed and filenames match the COCO annotations.")


FileNotFoundError: [Errno 2] No such file or directory: '../data/annotations/master_coco.json'

Inspect N_FRAMES randomly chosen annotated frames from one site

In [ ]:
TARGET_SITE = "H25"
N_FRAMES = 10
SEED = 5040
N_COLS = 1

# Uses by_site, gt_detections and images_dir from the sanity-check cell above
candidates = by_site[TARGET_SITE]
rng = random.Random(SEED)
sampled = rng.sample(candidates, min(N_FRAMES, len(candidates)))
stems = [Path(img["file_name"]).stem for img in sampled]
missing_files = [s for s in stems if not (images_dir / f"{s}.png").exists()]

check_annotator = VideoAnnotator(
    images_dir=str(images_dir),
    gt_detections=gt_detections,
    class_names=["bird"],
    image_files=[images_dir / f"{s}.png" for s in stems if s not in missing_files],
)

n_rows = max(1, -(-len(stems) // N_COLS))
fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 8, n_rows * 5), squeeze=False)

for i, stem in enumerate(stems):
    ax = axes[i // N_COLS][i % N_COLS]
    frame_path = images_dir / f"{stem}.png"

    if stem in missing_files:
        ax.text(0.5, 0.5, f"FILE NOT FOUND:\n{frame_path.name}",
                ha="center", va="center", transform=ax.transAxes, color="red", fontsize=9)
        ax.set_title(f"{stem} MISSING", color="red", fontsize=8)
        ax.axis("off")
        continue

    annotated_bgr = check_annotator.annotate_single_frame(str(frame_path))
    n_birds = len(gt_detections.get(stem, []))
    ax.imshow(annotated_bgr[:, :, ::-1])  # BGR -> RGB
    ax.set_title(f"{frame_path.name}\n{n_birds} bird(s)", fontsize=8)
    ax.axis("off")

# Hide unused axes
for j in range(len(stems), n_rows * N_COLS):
    axes[j // N_COLS][j % N_COLS].axis("off")

plt.tight_layout()
plt.show()

print(f"[{TARGET_SITE}] seed={SEED}: showing {len(stems)} of {len(candidates)} annotated frames")
if missing_files:
    print(f"Frames not found on disk ({len(missing_files)}):")
    for s in missing_files:
        print(f"  {s}")


## Prepare COCO Annotations

### Combine COCO annotations from different sites

In [ ]:
output_path = Path("../data/hlw/hlw_coco.json")

# Normalize both sites' categories to a single unified list
unified_categories = [{"id": 1, "name": "bird", "supercategory": ""}]

merged = {
    "info": {"description": "HLW combined dataset (H03 + H23)"},
    "licenses": [],
    "categories": unified_categories,
    "images": [],
    "annotations": [],
}

for site in sites:
    with open(coco_ann_files[site]) as f:
        coco = json.load(f)

    # Prefix filenames with site name
    for img in coco["images"]:
        new_img = copy.copy(img)
        new_img["file_name"] = f"{site}_{img['file_name']}"
        merged["images"].append(new_img)

    # Normalize category id to unified category (case-insensitive: 'bird'/'Bird' → 1)
    cat_id_to_unified = {cat["id"]: 1 for cat in coco["categories"]}
    for ann in coco["annotations"]:
        new_ann = copy.copy(ann)
        new_ann["category_id"] = cat_id_to_unified[ann["category_id"]]
        merged["annotations"].append(new_ann)

    print(f"[{site}] {len(coco['images'])} images, {len(coco['annotations'])} annotations")

with open(output_path, "w") as f:
    json.dump(merged, f)

print(f"\nSaved combined COCO annotations → {output_path}")
print(f"  Total images:      {len(merged['images'])}")
print(f"  Total annotations: {len(merged['annotations'])}")
print(f"  Categories:        {merged['categories']}")

### Remove Multi-Bird Frames

In [ ]:
coco_json_path = Path("../data/annotations/master_coco_multi.json")
coco_json_filtered_path = Path("../data/annotations/master_coco.json")

#remove_multi_bird_frames(coco_json_path, coco_json_filtered_path)

## Compute Video Statistics

In [3]:
coco_json_path = Path("../data/annotations/master_coco.json")
data_dir = "../data"
video_stats_df = compute_split_statistics(coco_json_path=coco_json_path)
display(video_stats_df)

,video,n_total_frames,n_bird_frames,prevalence,stratum
0,H03_IMG_0231,603,0,0.000000,0
1,H03_IMG_0507,303,55,0.181518,2
2,H03_IMG_0509,453,160,0.353201,2
3,H03_IMG_0511,303,23,0.075908,1
4,H03_IMG_0513,303,16,0.052805,1
...,...,...,...,...,...
261,H56_IMG_0008,453,262,0.578366,3
262,H56_IMG_0009,303,145,0.478548,2
263,H60_IMG_0005,303,35,0.115512,1
264,H60_IMG_0006,231,231,1.000000,3


## Split Dataset

In [4]:
target_split = {"Train": 0.70, "Val": 0.15, "Test": 0.15}

train_videos, val_videos, test_videos = site_video_split(df=video_stats_df, train_frac=target_split["Train"], val_frac=target_split["Val"], test_frac=target_split["Test"],
                                                         random_state=42, save_dir="../data/splits", n_trials=2000, density_weight=0.5)


split_report(video_stats_df, {"Train": train_videos, "Val": val_videos, "Test": test_videos}, targets=target_split)

Split saved to ../data/splits/split.json

Split  Target %  Actual %  Total frames  Annotated frames  % annotated
-----  --------  --------  ------------  ----------------  -----------
Train     70.0%     70.2%        55,446            13,596        24.5%
Val       15.0%     14.9%        11,804             7,264        61.5%
Test      15.0%     14.8%        11,726             3,689        31.5%
